# 03 — NUMA placement and socket scaling

**Question.** What does it cost to use memory on the other socket, how well do two sockets scale when run independently, and what happens when a single process spans both?

Remote and local refer to the memory socket relative to the CPU socket. Two-socket results use two processes timed over one shared clock window (at least 95% overlap required, below 99% is flagged) unless stated as a single spanning process.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench import report, roofline, plotting
from hwbench.notebook import setup
ctx = setup('03_numa_and_socket_scaling')
thesis, ext, theory = ctx.thesis, ctx.ext, ctx.theory
table, figure, say, say_claims = ctx.table, ctx.figure, ctx.say, ctx.say_claims
from hwbench import claims

## 1. Local against remote memory

**Conditions.** Flat mode, 56 cores of one socket, 8 GiB, memory bound to the local or the other socket's tier. Read-only and non-temporal Triad show how the penalty depends on traffic direction. Where the remote repeat (5 s windows, a local control beside every remote point) measured a pair, it replaces the first campaign; the *source* column says which.

In [ ]:
loc = report.locality_ratio(ext, thesis)
lr = report.local_remote_table(ext, thesis)
fig = plotting.go.Figure()
for placement, hatch in (('local', ''), ('remote', '/')):
    fig.add_bar(x=lr.label, y=lr[placement], showlegend=False,
                marker=dict(color=[plotting.CONFIG_COLORS[c] for c in lr.config], pattern=dict(shape=hatch, fgcolor=plotting.SURFACE, size=6, solidity=0.5), line=dict(width=0)),
                error_y=dict(type='data', array=lr[f'{placement}_spread'], color=plotting.INK_2, thickness=1.2))
plotting.style(fig, 'Local (solid) and remote (hatched) bandwidth · socket 0, bars show the difference to socket 1', 'Tier and kernel', 'GB/s', 480)
fig.update_layout(barmode='group', bargap=0.3, margin=dict(l=85, r=40, t=85, b=80))
figure(fig, 'local_remote_bandwidth')
table(lr[['label', 'local', 'remote', 'remote_over_local', 'local_spread', 'remote_spread', 'source']].rename(columns={'local_spread': 'local, |socket 1 − 0|', 'remote_spread': 'remote, |socket 1 − 0|'}), 2)

In [ ]:
say_claims(claims.placement(ext, thesis))

The first campaign measured remote bandwidth only at 56 threads. A separate repeat (5 s windows, a local control beside every remote point) sweeps the thread count:

In [ ]:
rt = report.remote_repeat_table(ext)
fig = plotting.remote_threads_figure(rt)
figure(fig, 'remote_hbm_threads')
table(rt[rt.socket == 0][['kernel', 'memory_target', 'threads', 'gbs_local', 'gbs_remote', 'cv_remote', 'remote_over_local']], 2)

Bind memory and CPUs to the same socket for bandwidth-bound work.

## 2. Two sockets, two processes

Each pair completes initialization and warm-up, then synchronizes readiness before a shared clock window. Efficiency is the pair's aggregate throughput divided by the sum of the isolated socket rates.

In [ ]:
de = report.dual_efficiency(ext, thesis)
bandwidth = de.benchmark.str.startswith('stream')
de['label'] = np.where(bandwidth,
    de.benchmark.str.replace('stream_', '', regex=False) + ' ' + de.config.str.replace('flat_', '', regex=False) + ' ' + de.working_set_gib.round(0).fillna(0).astype(int).astype(str) + 'G',
    de.benchmark.str.replace('_peak', '', regex=False) + ' ' + de.dtype.fillna(''))
view = de[de.working_set_gib.isna() | (de.working_set_gib <= 16)].copy()
view['colour'] = [plotting.CONFIG_COLORS[c] if b.startswith('stream') else plotting.PATH_COLORS[theory.isa_for(b, d).key]
                  for b, d, c in zip(view.benchmark, view.dtype, view.config)]
fig = plotting.dual_efficiency_figure(view)
figure(fig, 'dual_socket_efficiency')
table(de.drop(columns=['label', 'metric']))

In [ ]:
say(f"**Result.** Across {len(de)} paired measurements the two-process rate is {de.efficiency.min():.3f}–{de.efficiency.max():.3f}× the sum of isolated sockets "
    f"(median {de.efficiency.median():.3f}); the shared window overlapped by at least {de.overlap_min.min():.1%}.")
say_claims(claims.sockets(ext, thesis))

## 3. One process spanning both sockets

Unpinned workloads run one process with threads on all 112 cores. Two placements are compared for 16 GiB of data: *interleaved* pages across both sockets' memory, and *first-touch* (each thread allocates local to its socket).

In [ ]:
sp = report.span_table(ext)
table(sp[['memory_mode', 'config', 'benchmark', 'placement', 'k', 'gbs', 'gops', 'cv_percent', 'n_samples']])

In [ ]:
ref = pd.concat([report.span_reference(ext, thesis, 'read'), report.span_reference(ext, thesis, 'triad')], ignore_index=True)
fig = plotting.go.Figure()
for placement, hatch in (('firsttouch', ''), ('interleave', '/')):
    part = ref[ref.placement == placement]
    fig.add_bar(x=part.label, y=part.gbs, showlegend=False,
                marker=dict(color=[plotting.CONFIG_COLORS[c] for c in part.config], pattern=dict(shape=hatch, fgcolor=plotting.SURFACE, size=6, solidity=0.5), line=dict(width=0)))
once = ref.drop_duplicates('label')
fig.add_scatter(x=once.label, y=once.isolated_sum, mode='markers', name='two isolated sockets, summed',
                marker=dict(symbol='diamond', size=12, color=plotting.INK, line=dict(color=plotting.SURFACE, width=1.5)))
plotting.style(fig, 'One 112-thread process · first-touch (solid) and interleaved (hatched)', 'Tier and kernel', 'GB/s', 500)
fig.update_layout(barmode='group', bargap=0.3, margin=dict(l=85, r=40, t=85, b=110))
figure(fig, 'span_bandwidth')

In [ ]:
say_claims(claims.span(ext, thesis))

## 4. Cores needed to saturate one socket (October Triad)

In [ ]:
from hwbench.analysis import scaling_table, knees_table
core = thesis[(thesis.benchmark == 'stream_triad') & (thesis.notes == 'core_scaling') & (thesis.socket_scope == 'single') & (thesis.socket == 0) & (thesis.locality == 'local')]
c = core.groupby(['memory_mode', 'memory_target', 'threads']).bandwidth_gbs.median().reset_index()
c['config'] = [plotting.config_key(m, t) for m, t in zip(c.memory_mode, c.memory_target)]
fig = plotting.line_by_config(c, 'threads', 'bandwidth_gbs', 'config', 'Triad bandwidth against core count · socket 0, 8 GiB', 'Cores', 'GB/s')
figure(fig, 'triad_core_scaling')